# Lesson 4: Function Calling & Structured Outputs

In this lesson, you'll learn how to give AI the ability to call functions and produce structured data.

## Topics Covered
1. Understanding function calling
2. Defining tools for the AI
3. Handling function calls and results
4. Structured outputs with JSON mode

## Learning Objectives
- Understand when and why to use function calling
- Define functions that AI can call
- Handle the complete function calling flow
- Use structured outputs for reliable data extraction

In [ ]:
# Install required packages if not already installed
#%pip install python-dotenv
#%pip install openai

# Load environment variables from .env file
import os
from dotenv import load_dotenv
load_dotenv()
import openai
import json
from datetime import datetime
print("OpenAI package version:", openai.__version__)

In [ ]:
# Set up the OpenAI client with environment variables
chat_client = openai.OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=int(os.getenv("OPENAI_TIMEOUT", 30)),
    max_retries=int(os.getenv("MAX_RETRIES", 3)),
    base_url=os.getenv("OPENAI_ENDPOINT")    
)

print("Client configured successfully!")

## 1. Understanding Function Calling

**Function calling** lets AI decide when to use external functions/tools to get accurate information.

**Think of it like this:**
- **Without functions**: AI is like a student taking a closed-book exam - can only use memorized knowledge (sometimes wrong!)
- **With functions**: AI is like a student with a calculator and textbook - uses tools when needed for accuracy

**When to use function calling:**
- Math calculations (AI isn't reliable with large numbers)
- Current information (time, weather, stock prices)
- Database queries
- API calls
- Any task requiring real-time or precise data

In [ ]:
# Example 1A: Without function calling (unreliable for complex math)
response = chat_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "What is 8,247 multiplied by 6,891?"}
    ]
)

print("WITHOUT function calling:")
print(f"AI says: {response.choices[0].message.content}")
print(f"\nActual answer: {8247 * 6891:,}")
print("\n⚠️  AI might approximate or get complex math wrong!")
print("="*80 + "\n")

In [ ]:
# Example 1B: WITH function calling (always accurate)

# Step 1: Define a calculator function
def calculate(operation, num1, num2):
    """Perform basic arithmetic operations"""
    operations = {
        "add": num1 + num2,
        "subtract": num1 - num2,
        "multiply": num1 * num2,
        "divide": num1 / num2 if num2 != 0 else "Error: Division by zero"
    }
    return operations.get(operation, "Unknown operation")

# Step 2: Define the tool schema for AI
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform basic arithmetic operations (add, subtract, multiply, divide). Use this for any mathematical calculations to ensure accuracy.",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "The arithmetic operation to perform"
                    },
                    "num1": {
                        "type": "number",
                        "description": "First number"
                    },
                    "num2": {
                        "type": "number",
                        "description": "Second number"
                    }
                },
                "required": ["operation", "num1", "num2"]
            }
        }
    }
]

# Step 3: AI can now use the calculator
response = chat_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=[
        {"role": "user", "content": "What is 8,247 multiplied by 6,891?"}
    ],
    tools=tools,
    tool_choice="auto"  # Let AI decide when to use tools
)

print("WITH function calling:")
message = response.choices[0].message
print(f"First response: {message.content}")

# Check if AI wants to call a function
if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"✅ AI decided to use: {tool_call.function.name}")
    print(f"   With arguments: {tool_call.function.arguments}")
    
    # Parse arguments and execute function
    args = json.loads(tool_call.function.arguments)
    result = calculate(args["operation"], args["num1"], args["num2"])
    
    print(f"\n🎯 Result: {result:,}")
    print("   (Always accurate!)")
else:
    print(f"AI response: {message.content}")

## 2. Defining Tools for the AI

**Tool definition anatomy:**
```
{
  "type": "function",
  "function": {
    "name": "function_name",           // What to call
    "description": "When to use it",   // CRITICAL: Helps AI decide
    "parameters": {                    // JSON Schema format
      "type": "object",
      "properties": {...},
      "required": [...]
    }
  }
}
```

**Key principle:** The description tells the AI WHEN to use the tool. Be specific!

In [ ]:
# Example 2A: Multiple practical tools

# Tool 1: Get current time
def get_current_time():
    """Get current time and date"""
    now = datetime.now()
    return {
        "time": now.strftime("%I:%M %p"),
        "date": now.strftime("%Y-%m-%d"),
        "day": now.strftime("%A")
    }

# Tool 2: Get weather (simulated)
def get_weather(location, unit="celsius"):
    """Get current weather for a location (simulated data)"""
    weather_data = {
        "new york": {"temp": 22, "condition": "Partly cloudy", "humidity": 65},
        "london": {"temp": 15, "condition": "Rainy", "humidity": 80},
        "tokyo": {"temp": 28, "condition": "Sunny", "humidity": 55},
        "paris": {"temp": 18, "condition": "Cloudy", "humidity": 70}
    }
    
    location_lower = location.lower()
    if location_lower in weather_data:
        data = weather_data[location_lower]
        temp = data["temp"]
        
        # Convert to Fahrenheit if requested
        if unit == "fahrenheit":
            temp = (temp * 9/5) + 32
        
        return {
            "location": location,
            "temperature": round(temp, 1),
            "unit": unit,
            "condition": data["condition"],
            "humidity": data["humidity"]
        }
    return {"error": f"Weather data not available for {location}"}

# Tool 3: Search knowledge base (simulated)
def search_knowledge(query):
    """Search company knowledge base (simulated)"""
    knowledge = {
        "return": "Items can be returned within 30 days with receipt. Refund processed in 5-7 business days.",
        "shipping": "Free shipping on orders over $50. Standard delivery: 3-5 business days. Express: 1-2 days.",
        "warranty": "All products include 1-year manufacturer warranty. Extended warranties available at checkout.",
        "payment": "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay.",
        "servicehours": "Customer service available Mon-Fri 9am-6pm EST. Live chat 24/7."
    }
    
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return {"answer": value, "source": "Company Knowledge Base", "topic": key}
    
    return {"answer": "No information found. Please contact customer service.", "source": "Knowledge Base"}

print("✅ Tools defined:")
print("  1. get_current_time() - Get time and date")
print("  2. get_weather(location, unit) - Get weather for a city")
print("  3. search_knowledge(query) - Search company policies")

In [ ]:
# Example 2B: Define tool schemas for AI
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current time, date, and day of the week. Use this when user asks 'what time is it', 'what day is it', or needs current date/time information.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather conditions for a specific city. Use when user asks about weather, temperature, or conditions in a location. Available cities: New York, London, Tokyo, Paris.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name, e.g., 'New York', 'London', 'Tokyo', 'Paris'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit. Defaults to celsius."
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search company knowledge base for policies and information. Use when user asks about returns, shipping, warranty, payment methods, or customer service hours.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query or topic (e.g., 'return policy', 'shipping', 'warranty')"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("\n📋 Tool Schemas Defined:")
for tool in tools:
    func = tool["function"]
    print(f"\n  • {func['name']}")
    print(f"    Description: {func['description'][:80]}...")
    print(f"    Parameters: {list(func['parameters']['properties'].keys()) if func['parameters']['properties'] else 'None'}")

In [ ]:
# Example 2C: AI chooses the right tool
def test_tool_selection(user_query):
    """Test which tool AI selects for different queries"""
    print(f"\n{'='*80}")
    print(f"🧑 User: {user_query}")
    
    response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[{"role": "user", "content": user_query}],
        tools=tools,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        print(f"\n🤖 AI chose: {tool_call.function.name}")
        print(f"   Arguments: {tool_call.function.arguments}")
    else:
        print(f"\n🤖 AI response: {message.content}")
        print("   (No tool needed)")

# Test different queries
test_tool_selection("What's the weather like in Tokyo?")
test_tool_selection("What's your return policy?")
test_tool_selection("What time is it?")
test_tool_selection("Tell me a joke")  # Should NOT use tools

## 3. Handling Function Calls and Results

**The complete function calling flow:**

1. **User asks a question** → Send to AI with tools
2. **AI decides** → Use tool or respond directly
3. **If tool needed** → AI returns tool call request
4. **You execute** → Run the actual function
5. **Send result back** → AI uses it to form final answer

Think of it like a relay race: User → AI → Function → AI → User

In [ ]:
# Example 3A: Complete function calling flow (step by step)
user_message = "What's the temperature in London in Fahrenheit?"

print("=" * 80)
print("COMPLETE FUNCTION CALLING FLOW")
print("=" * 80)
print(f"\n🧑 User: {user_message}\n")

# STEP 1: Send message to AI with available tools
print("[Step 1] Sending to AI with tools...")
messages = [{"role": "user", "content": user_message}]

response = chat_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"✓ AI responded")

# STEP 2: Check if AI wants to use a tool
if message.tool_calls:
    print(f"\n[Step 2] AI wants to call a function")
    
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    print(f"   Function: {function_name}")
    print(f"   Arguments: {arguments}")
    
    # STEP 3: Execute the actual function
    print(f"\n[Step 3] Executing function...")
    
    if function_name == "get_weather":
        result = get_weather(**arguments)
    elif function_name == "get_current_time":
        result = get_current_time()
    elif function_name == "search_knowledge":
        result = search_knowledge(**arguments)
    else:
        result = {"error": "Unknown function"}
    
    print(f"   Result: {result}")
    
    # STEP 4: Add function result to conversation
    print(f"\n[Step 4] Sending result back to AI...")
    messages.append(message)  # Add AI's tool call message
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(result)
    })
    
    # STEP 5: AI uses result to form final response
    print(f"\n[Step 5] AI forming final response...")
    final_response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=messages
    )
    
    final_message = final_response.choices[0].message.content
    print(f"\n🤖 Assistant: {final_message}")
else:
    print(f"\n🤖 Assistant: {message.content}")
    print("   (No tool needed)")

In [ ]:
# Example 3B: Reusable function calling helper
def chat_with_tools(user_message, tools_list, verbose=False):
    """Complete function calling flow in one function"""
    if verbose:
        print(f"\n🧑 You: {user_message}\n")
    
    messages = [{"role": "user", "content": user_message}]
    
    # Step 1: Get AI response
    response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=messages,
        tools=tools_list,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    messages.append(message)
    
    # Step 2: If tool calls, execute them
    if message.tool_calls:
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            if verbose:
                print(f"🔧 Calling: {function_name}({arguments})")
            
            # Execute function
            if function_name == "get_weather":
                result = get_weather(**arguments)
            elif function_name == "get_current_time":
                result = get_current_time()
            elif function_name == "search_knowledge":
                result = search_knowledge(**arguments)
            else:
                result = {"error": f"Unknown function: {function_name}"}
            
            if verbose:
                print(f"   Result: {result}\n")
            
            # Add result to conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
        
        # Step 3: Get final response from AI
        final_response = chat_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            messages=messages
        )
        
        final_message = final_response.choices[0].message.content
        if verbose:
            print(f"🤖 Assistant: {final_message}")
        return final_message
    else:
        if verbose:
            print(f"🤖 Assistant: {message.content}")
        return message.content

# Test the helper function
print("=" * 80)
print("Using Helper Function")
print("=" * 80)

chat_with_tools("Can you check the weather in Paris?", tools, verbose=True)
print("\n" + "="*80 + "\n")
chat_with_tools("What's your shipping policy?", tools, verbose=True)

## 4. Structured Outputs with JSON Mode

**JSON mode** ensures AI always returns valid, parseable JSON.

**Use cases:**
- Extracting structured data from text
- Form filling from natural language
- Database updates
- API integrations
- Data analysis pipelines

**Analogy:** Function calling is like giving AI a calculator. JSON mode is like giving AI a standardized form to fill out - the format is always consistent and predictable.

 **Use cases:**
   - Extracting data from text
   - Form filling
   - Database updates
   - API integrations

In [ ]:
import re

# Example 4A: Basic JSON mode
print("=" * 80)
print("Extracting Contact Information as JSON")
print("=" * 80)

# Sample text to extract contact info from
text = (
    "You can reach me at john.doe@example.com or at work-email@company.com. "
    "Please call (555) 123-4567 or +1-555-765-4321 for urgent matters. "
    "Office address: 123 Main St, Springfield, IL 62704. "
    "Website: https://example.com. LinkedIn: https://linkedin.com/in/johndoe"
)

# Ask the model to return ONLY valid JSON that matches the schema:
# {
#   "name": optional string,
#   "emails": array of strings,
#   "phones": array of strings,
#   "addresses": array of strings,
#   "websites": array of strings
# }
messages = [
    {
        "role": "system",
        "content": (
            "You are a JSON extractor. Given some text, extract contact information and "
            "return ONLY valid JSON (no explanatory text). The JSON MUST follow this schema:\n\n"
            "{\n  \"name\": string | null,\n  \"emails\": [string],\n  \"phones\": [string],\n  \"addresses\": [string],\n  \"websites\": [string]\n}\n\n"
            "If a field is not present, return null for name and empty arrays for lists."
        )
    },
    {
        "role": "user",
        "content": f"Extract the contact information from the following text as JSON:\n\n\"{text}\""
    }
]

response = chat_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    messages=messages,
    # keep deterministic output
    temperature=0
)

raw = response.choices[0].message.content

# Try to robustly extract JSON in case the model adds surrounding backticks or text
try:
    # Find the first JSON object in the response
    m = re.search(r"\{[\s\S]*\}", raw)
    json_str = m.group(0) if m else raw.strip()
    extracted = json.loads(json_str)
except Exception as e:
    print("Failed to parse JSON from model output. Raw output:")
    print(raw)
    raise

# Pretty-print the extracted JSON and basic checks
print("\nExtracted JSON:")
print(json.dumps(extracted, indent=2))

# Example: access fields
emails = extracted.get("emails", [])
phones = extracted.get("phones", [])
addresses = extracted.get("addresses", [])

print("\nSummary:")
print(f"  emails: {emails}")
print(f"  phones: {phones}")
print(f"  addresses: {addresses}")

In [ ]:
# Example 4B: Structured data extraction with schema
def extract_structured_data(text, schema_description):
    """Extract data according to a specific schema"""
    response = chat_client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": f"""Extract information and return as JSON.
            
Schema:
{schema_description}

Return ONLY valid JSON matching this schema."""},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    
    return json.loads(response.choices[0].message.content)

# Test with customer feedback
feedback_text = """The product quality is excellent, but shipping took forever! 
Ordered on Jan 15th, received Feb 3rd. Customer service was helpful though.
Rating: 3/5 stars."""

schema = """
{
  "sentiment": "positive/neutral/negative",
  "rating": number (1-5),
  "aspects": {
    "product_quality": "positive/neutral/negative",
    "shipping_speed": "positive/neutral/negative",
    "customer_service": "positive/neutral/negative"
  },
  "order_date": "YYYY-MM-DD or null",
  "delivery_date": "YYYY-MM-DD or null"
}
"""

print("Extracting structured feedback data...\n")
result = extract_structured_data(feedback_text, schema)
print(json.dumps(result, indent=2))

print(f"\n📊 Quick insights:")
print(f"  Overall sentiment: {result['sentiment']}")
print(f"  Rating: {result['rating']}/5")
print(f"  Delivery timeline: Order date: {result['order_date']} → Delivery date:{result['delivery_date']}")

In [ ]:
# Example 3C: Batch data processing
def process_invoice_text(invoice_text):
    """Extract invoice data for accounting system"""
    schema = """
    {
      "invoice_number": "string",
      "date": "YYYY-MM-DD",
      "vendor": "string",
      "items": [
        {
          "description": "string",
          "quantity": number,
          "unit_price": number,
          "total": number
        }
      ],
      "subtotal": number,
      "tax": number,
      "total": number
    }
    """
    
    return extract_structured_data(invoice_text, schema)

# Sample invoice
invoice = """
INVOICE #INV-2024-001
Date: January 15, 2024
From: Office Supplies Co.

Items:
- Printer Paper (5 reams) @ $8.99 each = $44.95
- USB Cables (10 pack) @ $15.99 = $15.99
- Desk Lamp @ $35.00 = $35.00

Subtotal: $95.94
Tax (8%): $7.68
Total: $103.62
"""

print("Processing invoice...\n")
invoice_data = process_invoice_text(invoice)
print(json.dumps(invoice_data, indent=2))

print(f"\n💰 Total amount: ${invoice_data['total']}")
print(f"📦 Items: {len(invoice_data['items'])}")